# NB-13 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-13**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-13.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This applies the nodal-market representation to actual PJM hub prices, then calculates descriptive gross congestion spreads. It does not establish an awarded FTR position or net realized profit.

- [Volume V, Chapter 12 — A North American nodal market](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c12)
- [Volume VI, Chapter 10 — FTR settlement as a probabilistic network target](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c10)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-13', require_key=True)
print(setup_status)


# NB-13 — Actual PJM components to gross path settlements
Companion to Volume V §12 and Volume VI §10. Read actual DA observations for
Eastern Hub (51217) and Western Hub (51288), September 1–7, 2026, UTC.
Learn to reconcile components, align source/sink intervals, and integrate
a congestion spread. This retained latest-revision analysis is descriptive.

In [ ]:
import copy
from datetime import datetime
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot, read_saved
session = Session('NB-13', version='1.43.0')
receipt = session.inputs['source_receipt']
component_request = session.inputs['components']
settlement_request = session.inputs['settlement']
print('Local input inspection:', component_request['inputs']['lmp_component_panel'][:2])
components = session.execute('lmp_component_reconciliation', component_request)['result']['component_reconciliation']
assert len(components['rows']) == 336
assert all(row['within_tolerance'] for row in components['rows'])
assert all(row['energy_common'] for row in components['spreads'])
settlement_response = session.execute('ftr_product_settlement', settlement_request)
settlement = settlement_response['result']['gross_product_cash']
times = [datetime.fromisoformat(t) for t in settlement['starts_utc']]
ledger = [{'start_utc':r['start'], 'duration_hours':h, 'congestion_price':r['congestion'],
    'marginal_loss_price':r['loss'], 'total_lmp':r['total']} for r,h in zip(
    components['spreads'],settlement_request['inputs']['congestion_price_panel']['durations_hours'])]
assert len(ledger)==len(times)==168
print({'snapshot':receipt['snapshot_id'], 'rows':len(components['rows']),'hours':len(times),
    'source':51217, 'sink':51288, 'policy':receipt['information_policy']})
print('First interval:', ledger[0])

## Keep the three spreads separate
The nominal gross congestion payoff uses sink-minus-source congestion
components. Total LMP includes losses too. System energy cancels between
these two nodes at each shared hour. The API end bound is inclusive;
the adapter excludes September 8 at 00:00 to make a 168-hour window.

In [ ]:
congestion = np.array([r['congestion_price'] for r in ledger])
loss = np.array([r['marginal_loss_price'] for r in ledger])
total = np.array([r['total_lmp'] for r in ledger])
duration = np.array([r['duration_hours'] for r in ledger])
np.testing.assert_allclose(total,congestion+loss,atol=1e-5)
# A per-MW payoff is a unit normalization, not an observed trading position.
gross_per_mw = float(congestion @ duration)
print({'gross_congestion_usd_per_mw':gross_per_mw,
       'loss_spread_usd_per_mw':float(loss @ duration),
       'total_price_spread_usd_per_mw':float(total @ duration),
       'negative_congestion_hours':int((congestion<0).sum())})
plt=setup_plot()
fig, ax=plt.subplots()
ax.plot(times,congestion,label='Congestion',color='#88C0D0')
ax.plot(times,total,label='Total LMP spread',color='#EBCB8B',alpha=.8)
ax.axhline(0,color='#4C566A'); ax.set(ylabel='USD/MWh',title='Western minus Eastern Hub · DA · UTC')
legend=ax.legend(facecolor='#3B4252',labelcolor='#ECEFF4')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## Exercise: reverse the path
Calculate Eastern-minus-Western directly from the underlying rows and check
that every payoff changes sign. This property belongs to obligations;
flooring the spread at zero would define a different payoff.

In [ ]:
reverse_request = copy.deepcopy(settlement_request)
paths = reverse_request['inputs']['path_catalog']
paths['sources'], paths['sinks'] = paths['sinks'], paths['sources']
reverse_result = session.execute('ftr_product_settlement', reverse_request)['result']['gross_product_cash']
reverse = np.asarray(reverse_result['spread_per_mwh'])[:,0]
np.testing.assert_allclose(reverse,-congestion)
assert abs(reverse_result['gross_cash'][0]+gross_per_mw)<1e-7
# Remove the same acquired observation from the typed input to test alignment.
incomplete = copy.deepcopy(component_request)
incomplete['inputs']['lmp_component_panel'].pop(0)
try:
    session.execute('lmp_component_reconciliation', incomplete)
except MethodsError as exc:
    assert 'HTTP 422' in str(exc)
    print('Expected missing-hour rejection:',exc)
else:
    raise AssertionError('Incomplete panel accepted')

In [ ]:
np.testing.assert_allclose(settlement['gross_cash'][0],gross_per_mw,atol=1e-7)
summary = session.finish(
    ['336 unique observations','168 aligned hours','component sums','path reversal','missing hour rejected'],
    {'source_pnode':51217,'sink_pnode':51288,'product':'all hours in declared UTC week','information_policy':'final_descriptive'})
print(summary)
saved = read_saved(session.output / session.calls[1]['result_file'])
assert saved['receipt']['result_sha256'] == settlement_response['receipt']['result_sha256']

## What failed, and what the payoff leaves out
A missing hour stops the calculation; it is not a zero spread. This is a
nominal gross congestion payoff per MW over a declared UTC window, not
a tariff-certified FTR product or funded cash return. Auction acquisition
costs, fees, funding, effective-dated membership, and historical predictor
availability are not supplied. Prices do not reconstruct physical PTDFs.